# Exercise: Building a Fair and Ethical AI System
## AIAT — Generative AI | Unit 4: Ethics & Regulations

## Learning Objectives

After completing this exercise, you will be able to:
- Detect bias in a real dataset and measure its impact on model fairness
- Apply fairness-aware training techniques (reweighting, threshold adjustment)
- Implement transparency tools (feature importance, prediction explanations)
- Audit a model against GDPR / EU AI Act compliance checklist
- Produce a structured AI Ethics Report

## Real-World Context

You are a responsible AI engineer at a bank. The loan-approval model has been flagged for **disparate impact** — it approves loans at a lower rate for applicants from certain demographic groups. Your job is to diagnose the bias, apply a fix, verify fairness improved, and document everything for the compliance team.

**Theory → Practice:** Review the examples:
- `building_ethical_ai_models_with_principles_like_fairness_and_transparency.ipynb`
- `applying_ai_regulatory_guidelines_like_gdpr_to_ensure_compliance_in_model_develo.ipynb`

---

## 📥 Inputs & 📤 Outputs

**Inputs:** Synthetic loan dataset (generated in-notebook), a trained biased classifier

**Outputs:** Fairness metrics before/after fix, feature importance chart, GDPR compliance table, Ethics Report

**Expected when complete:** You should see the demographic parity gap shrink after your fix, a bar chart of feature importances, and a printed compliance audit with all items checked.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ── Generate synthetic loan dataset ────────────────────────────────────────
N = 2000
group = np.random.choice([0, 1], size=N, p=[0.6, 0.4])   # 0 = majority, 1 = minority
income   = np.random.normal(50_000 + group * (-8_000), 15_000, N).clip(15_000, 150_000)
debt     = np.random.normal(0.35 - group * 0.05, 0.10, N).clip(0.05, 0.95)
credit   = np.random.normal(650 + group * (-30), 80, N).clip(300, 850)

# Biased label: minority group has 15% lower approval regardless of financials
prob_approve = 0.3 + 0.4 * (income / 150_000) - 0.3 * debt + 0.2 * ((credit - 300) / 550)
prob_approve -= 0.15 * group   # <── injected bias
prob_approve = prob_approve.clip(0.05, 0.95)
label = (np.random.rand(N) < prob_approve).astype(int)

X = np.column_stack([income, debt, credit, group])
feature_names = ['income', 'debt_ratio', 'credit_score', 'group']
y = label

X_train, X_test, y_train, y_test, g_train, g_test = train_test_split(
    X, y, group, test_size=0.3, random_state=42
)

print('Dataset ready.')
print(f'  Training samples : {len(X_train)}')
print(f'  Test samples     : {len(X_test)}')
print(f'  Approval rate (majority group): {y_test[g_test==0].mean():.2%}')
print(f'  Approval rate (minority group): {y_test[g_test==1].mean():.2%}')

---
## Task 1: Train a Baseline Model and Measure Bias (25 points)

Train a Logistic Regression classifier on all features **including** `group`.  
Then compute the **demographic parity difference** (approval rate gap between groups).

**Demographic Parity Difference** = |P(approve | group=0) − P(approve | group=1)|

A value > 0.10 is typically flagged as discriminatory bias.

In [ ]:
# TODO 1a: Train a LogisticRegression model on X_train, y_train
# Hint: LogisticRegression(max_iter=1000)
model_biased = None  # <-- replace with your code

# TODO 1b: Predict on X_test
y_pred_biased = None  # <-- replace with your code

# TODO 1c: Compute overall accuracy
acc_biased = None  # <-- replace with your code

# TODO 1d: Compute approval rate for each group separately
rate_majority = None  # P(predict=1 | group=0)
rate_minority = None  # P(predict=1 | group=1)

# TODO 1e: Compute demographic parity difference
dp_diff = None  # |rate_majority - rate_minority|

print(f'Accuracy         : {acc_biased:.2%}')
print(f'Approval (majority): {rate_majority:.2%}')
print(f'Approval (minority): {rate_minority:.2%}')
print(f'Demographic Parity Difference: {dp_diff:.2%}')
if dp_diff and dp_diff > 0.10:
    print('⚠️  BIAS DETECTED — gap exceeds 10% threshold!')

---
## Task 2: Apply a Fairness Fix — Sample Reweighting (25 points)

One simple fix is to **remove the protected attribute** from features and apply **sample weights** that upweight the minority group during training.  
This forces the model to give equal attention to both groups.

Steps:
1. Drop the `group` column from X (index 3)
2. Compute per-sample weights: minority gets weight = (N_majority / N_minority), majority gets weight = 1.0
3. Retrain with `sample_weight` parameter
4. Measure how much the demographic parity difference improved

In [ ]:
# TODO 2a: Drop the group column from train and test sets
X_train_fair = None  # X_train without column index 3
X_test_fair  = None  # X_test  without column index 3

# TODO 2b: Compute sample weights
# minority_count = number of training samples where g_train == 1
# majority_count = number of training samples where g_train == 0
# weight for minority = majority_count / minority_count
# weight for majority = 1.0
sample_weights = None  # numpy array of shape (len(X_train),)

# TODO 2c: Train a new LogisticRegression with sample_weight
model_fair = None  # <-- your code

# TODO 2d: Predict and compute metrics
y_pred_fair = None
acc_fair = None
rate_majority_fair = None
rate_minority_fair = None
dp_diff_fair = None

print('=== After Fairness Fix ===')
print(f'Accuracy         : {acc_fair:.2%}')
print(f'Approval (majority): {rate_majority_fair:.2%}')
print(f'Approval (minority): {rate_minority_fair:.2%}')
print(f'Demographic Parity Difference: {dp_diff_fair:.2%}')
print(f'\nImprovement: {dp_diff:.2%} → {dp_diff_fair:.2%}')

---
## Task 3: Transparency — Feature Importance (20 points)

Regulators (GDPR Article 22) require that automated decisions can be **explained**.  
Use the model coefficients from your fair model to visualize which features matter most.

Then write 2–3 sentences interpreting what the chart tells you.

In [ ]:
# TODO 3a: Extract coefficients from model_fair
# model_fair.coef_[0]  →  array of shape (3,) since we dropped group column
fair_feature_names = ['income', 'debt_ratio', 'credit_score']
coefs = None  # <-- your code

# TODO 3b: Plot a horizontal bar chart of feature importance (absolute value of coefs)
# Use plt.barh() and label each bar with the feature name
# Title: 'Feature Importance (Fair Model)'

# YOUR PLOT CODE HERE

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=80)
plt.close()
print('Chart saved.')

# TODO 3c: Print your interpretation
# Which feature has the highest importance? Is that expected? Why?

---
## Task 4: GDPR / EU AI Act Compliance Audit (15 points)

High-risk AI systems (like loan decisions) must satisfy several legal requirements.  
Complete the audit checklist below by filling in **True/False** and a one-line **evidence** note.

In [ ]:
# TODO 4: Fill in the compliance checklist
# For each item, set satisfied = True or False, and write a short evidence string

checklist = [
    {
        'requirement': 'GDPR Art.22 — Human oversight of automated decisions',
        'satisfied': None,    # True / False
        'evidence': '???'     # e.g. 'Loan officer reviews every rejection'
    },
    {
        'requirement': 'GDPR Art.22 — Right to explanation for the applicant',
        'satisfied': None,
        'evidence': '???'     # e.g. 'Feature importance shown to user via API'
    },
    {
        'requirement': 'EU AI Act Art.9 — Bias testing before deployment',
        'satisfied': None,
        'evidence': '???'     # e.g. 'Demographic parity checked — gap < 5%'
    },
    {
        'requirement': 'EU AI Act Art.13 — Transparency documentation',
        'satisfied': None,
        'evidence': '???'     # e.g. 'Model card written with training data description'
    },
    {
        'requirement': 'UNESCO Rec. 2021 — Protected attributes removed from features',
        'satisfied': None,
        'evidence': '???'     # e.g. 'Group column dropped from X_train_fair'
    },
]

print('=' * 60)
print('AI ETHICS COMPLIANCE AUDIT')
print('=' * 60)
for item in checklist:
    status = '✅' if item['satisfied'] else '❌'
    print(f"{status} {item['requirement']}")
    print(f"   Evidence: {item['evidence']}")
    print()

---
## Task 5: Ethics Report (15 points)

Write a short Ethics Report (3–5 sentences per section) in the cell below.  
This mirrors what a responsible AI team would submit to a regulator.

### Your Ethics Report

**1. Problem Statement**  
*(What bias was found? How was it measured?)*  
> TODO: Write here

**2. Root Cause Analysis**  
*(Why does the bias exist? Training data? Label collection? Feature selection?)*  
> TODO: Write here

**3. Mitigation Applied**  
*(What did you do to reduce the bias? Did it work?)*  
> TODO: Write here

**4. Residual Risk**  
*(Is the model perfectly fair now? What risk remains?)*  
> TODO: Write here

**5. Monitoring Plan**  
*(How would you detect if bias reappears in production?)*  
> TODO: Write here

---
## 📝 Summary

In this exercise you:
- Measured **demographic parity** as a concrete fairness metric
- Applied **sample reweighting** to reduce bias while preserving accuracy
- Used **model coefficients** to explain predictions (transparency)
- Audited the system against **GDPR / EU AI Act** requirements
- Produced a structured **Ethics Report** — a real deliverable in industry

**Key takeaway:** Fairness is not automatic — it must be actively measured, fixed, and documented throughout the ML lifecycle.

---
## 📚 References

- [GDPR Article 22 — Automated Decision Making](https://gdpr-info.eu/art-22-gdpr/)
- [EU AI Act (2024)](https://artificialintelligenceact.eu/)
- [Fairlearn — Microsoft Fairness Toolkit](https://fairlearn.org/)
- [Google ML Fairness](https://developers.google.com/machine-learning/fairness-overview)
- [What is Demographic Parity? — IBM](https://www.ibm.com/topics/ai-fairness)